## Imports

In [38]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
from langchain_core.tools import tool                                                                                                                                                                         
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage   
from groq import Groq
import re
import sqlite3
from typing import Tuple, List, Dict, Any
from ast import pattern
from re import search
import json

## Testando a conexão com o modelo

In [39]:
load_dotenv()

client = Groq()
completion = client.chat.completions.create(
    model = "openai/gpt-oss-120b",
    messages = [
        {
            "role": "user",
            "content": ""
        }
    ],
    stream = True,
    top_p = 1,
    stop = None,
    temperature = 0
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

It looks like your message didn’t come through. Could you let me know how I can help you today?

## Implementação de GuardRails

In [40]:
BANNED_KEYWORDS = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER",                                                                                                                                                              
                   "CREATE", "ATTACH", "DETACH", "PRAGMA", "REPLACE", "TRUNCATE"]

def validade_query(query: str) -> Tuple[bool, str]:
    """
    Valida se a query digitada pelo usuario é segura e somente de leitura.
    Retorna (True, "OK") ou (False, "Motivo de bloqueio").
    """
    cleaned_query = query.strip()
    
    if cleaned_query.endswith(";"):
        cleaned_query = cleaned_query[:-1].strip()
        
        
    if ";" in cleaned_query:
        return False, "Consultas Múltiplas não são permitidas"
    
    query_upper = cleaned_query.upper()
    query_sem_strings = re.sub(r"'(?:''|[^'])*'", "''", query_upper)
    
    if ";" in query_sem_strings:                                                                                                                                                                              
            return False, "Consultas Múltiplas não são permitidas" 
    
    if not (query_upper.startswith("WITH") or query_upper.startswith("SELECT")):
        return False, "A consulta deve obrigatoriamente ser um SELECT ou WITH."
    
    for keyword in BANNED_KEYWORDS:
        pattern = rf"\b{keyword}\b"
        if re.search(pattern,query_upper):
            return False, f"Comando proibido {keyword} detectado."
        
    return True, "OK"

In [41]:
def enforce_query_limits(query: str, default_limit: int = 100) -> str:
    """
    Inpeciona a Query e injeta um limit caso o da LLM não tenha sido definido.
    """
    
    cleaned_query = query.strip()
    if cleaned_query.endswith(";"):
        cleaned_query = cleaned_query[:-1].strip()
        
    query_upper = cleaned_query.upper()
    
    if re.search(r"\bLIMIT\b", query_upper):
        return cleaned_query
    else:
        return f"{cleaned_query} LIMIT {default_limit}"

In [42]:
def execute_safe_query(db_path: str, query: str) -> Dict[str, Any]:
    """
    Executa a query com todos os guardrails ativos.
    Retorna um  dicionário com:
    - sucess: bool
    - data: lista de registros ou erro
    - row_count: qtd de linhas
    """
    
    is_valid, reason = validade_query(query)
    if not is_valid:
        return {"success": False, "error": f"Guardrail Bloqueou: {reason}", "data": []}
    
    safe_query = enforce_query_limits(query)
    
    abs_path = os.path.abspath(db_path)
    
    uri_path = f"file:{abs_path}?mode=ro"
    
    try:
        conn = sqlite3.connect(uri_path, uri=True)
        cursor = conn.cursor()
        
        cursor.execute("PRAGMA query_only = ON")
        
        cursor.execute(safe_query)
        
        columns = [col[0] for col in cursor.description] if cursor.description else []                                                                                                                        
        rows = cursor.fetchall()  
        
        results = [dict(zip(columns, row)) for row in rows]
        
        conn.close()
        
        return {
            "success": True,                                                                                                                                                                                  
            "data": results,                                                                                                                                                                                  
            "row_count": len(results),                                                                                                                                                                        
            "executed_query": safe_query
        }
    except sqlite3.Error as e:
        return {"success": False, "error": f"Erro SQLite: {str(e)}", "data": []}

## Conectando ao LangChain

In [43]:
DB_PATH = "data/cinerocket.db"

@tool
def describe_tables(table_names: str) -> str:
    """
    Inspeciona o esquema do banco CineData.                                                                                                                                                                   
    Passe 'list' para ver os nomes de todas as tabelas disponíveis.                                                                                                                                           
    Ou passe nomes de tabelas separados por vírgula (ex: 'dim_movies, fact_movies_performance')                                                                                                               
    para ver suas colunas e tipos de dados.
    """
    
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    
    cleaned = table_names.strip().lower()
    
    if cleaned == "list":
        cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [r[0] for r in cur.fetchall() if not r[0].startswith("alembic")]                                                                                                                             
        conn.close()                                                                                                                                                                                          
        return f"Tabelas disponíveis: {', '.join(tables)}"
    
    output = []
    for table in table_names.split(","):
        table_clean = table.strip()
        cur.execute(f"PRAGMA table_info({table_clean})")
        cols = [f"{col[1]} ({col[2]})" for col in cur.fetchall()]                                                                                                                                             
        if cols:                                                                                                                                                                                              
            output.append(f"Tabela {table_clean}:\n  - " + "\n  - ".join(cols))                                                                                                                               
        else:                                                                                                                                                                                                 
            output.append(f"Tabela '{table_clean}' não encontrada no banco.")
            
    conn.close()
    return "\n\n".join(output)

In [44]:
@tool
def execute_sql_query(query: str) -> str:
    """
    Executa uma consulta SQL analítica no banco SQLite do CineData.                                                                                                                                           
    A query deve ser estritamente de leitura (SELECT ou WITH) e é protegida por guardrails.                                                                                                                   
    Retorna os dados em formato JSON ou a mensagem de erro detalhada para correção. 
    """
    result = execute_safe_query(DB_PATH, query)
    
    if not result["success"]:
        return f"ERRO NA CONSULTA: {result["error"]}"
    
    if not result["data"]:
        return "Consulta executada com sucesso, mas nenhum registro foi retornado com esses criterios."
    
    return json.dumps(result["data"][:15], ensure_ascii=False, default=str)

## Configuração do modelo e das Tools

In [45]:
tools = [describe_tables, execute_sql_query]

tools_by_name = {t.name: t for t in tools}

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

llm_with_tools = llm.bind_tools(tools)

## System Prompt

In [46]:
SYSTEM_PROMPT = """Você é o Agente Analítico Especialista do CineData Analytics.                                                                                                                              
    Sua missão é responder perguntas de negócio consultando a camada Gold no banco SQLite.                                                                                                                        
                                                                                                                                                                                                                  
    DIRETRIZES DE ARQUITETURA E EXECUÇÃO:                                                                                                                                                                         
    1. Sempre verifique o esquema das tabelas com `describe_tables` antes de chutar nomes de colunas.                                                                                                             
    2. A tabela fato central de métricas financeiras e notas é `fact_movies_performance`.                                                                                                                         
    3. A tabela dimensão com títulos e datas é `dim_movies`. Ambas se unem por `sk_movie_id`.                                                                                                                     
    4. Para atores e diretores, use `dim_people` ligada pela tabela bridge `bridge_movie_person`.                                                                                                                 
    5. Gere apenas SQL válido e utilize a ferramenta `execute_sql_query` para buscar os dados reais.                                                                                                              
    6. Sempre responda ao usuário em português brasileiro claro, objetivo e executivo.                                                                                                                            
    """

## Agent Loop

In [47]:
from urllib import response


def run_cinedata_agente(question: str, max_steps: int = 5, verbose: bool = 5) -> str:
    """
    Executa o ciclo de conversa do agente com o tool-calling e auto-correcao.
    """
    
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=question)
    ]
    
    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        if not response.tool_calls:
            return response.content
        
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]                                                                                                                                                                     
            tool_args = tool_call["args"]                                                                                                                                                                     
            tool_id = tool_call["id"] 
            
            if verbose:
                print(f"[Passo {step+1}] O agente chamou essa tool: {tool_name} com esses argumentos: {tool_args}")
                
            tool_obj = tools_by_name[tool_name]
            tool_output = tool_obj.invoke(tool_args)
            
            tool_message = ToolMessage(
                content=str(tool_output),
                tool_call_id = tool_id
            )
            
            messages.append(tool_message)
            
    return "O limite maximo de iteracoes sem uma resposta boa foi atingido :("

## Perguntas e testes para o agente

### 0.0: Pergunta Teste

In [48]:
pergunta = "Quantos filmes temos no catálogo ao todo e quais são os 3 filmes com maior lucro em reais?"                                                                                                       
                                                                                                                                                                                                                  
resposta = run_cinedata_agente(pergunta)                                                                                                                                                                       
print("\n\n")                                                                                                                                                                                          
print("RESPOSTA DO AGENTE:")                                                                                                                                                                                  
print(resposta)

[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': "WITH total AS (\n    SELECT COUNT(*) AS total_filmes FROM dim_movies\n), top3 AS (\n    SELECT d.titulo, f.lucro_brl\n    FROM dim_movies d\n    JOIN fact_movies_performance f USING (sk_movie_id)\n    ORDER BY f.lucro_brl DESC\n    LIMIT 3\n)\nSELECT (SELECT total_filmes FROM total) AS total_filmes,\n       json_group_array(json_object('titulo', titulo, 'lucro_brl', lucro_brl)) AS top3_filmes\nFROM top3;"}



RESPOSTA DO AGENTE:
**Total de filmes no catálogo:** **95 645**

**Top 3 filmes com maior lucro (em reais):**

| # | Filme | Lucro (BRL) |
|---|------------------------------|-------------------|
| 1 | Avatar: The Way Of Water | **9 933 736 500,54** |
| 2 | Spider‑Man: No Way Home | **9 835 362 882,74** |
| 3 | Avengers: Endgame | **9 684 105 600,00** |

Esses valore

### 1. Análise de Bilheteria e Finanças

#### 1.1: Top 10 filmes com maior receita em R$

In [49]:
pergunta = "Quais são os 10 filmes com maior receita em reais (receita_brl) no catálogo?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'fact_movies_performance, dim_movies'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT d.titulo, f.receita_brl\nFROM fact_movies_performance f\nJOIN dim_movies d ON f.sk_movie_id = d.sk_movie_id\nWHERE f.receita_brl IS NOT NULL\nORDER BY f.receita_brl DESC\nLIMIT 10;'}



RESPOSTA DO AGENTE:
Segue a lista dos **10 filmes com maior receita em reais (BRL)** no catálogo:

| Rank | Filme | Receita (BRL) |
|------|-------------------------------|-------------------|
| 1 | Avatar: The Way Of Water | **12 390 136 500,54** |
| 2 | Avengers: Endgame | **11 094 720 000,00** |
| 3 | Spider‑man: No Way Home | **10 977 782 882,74** |
| 4 | Avengers: Infinity War | **7 190 430 846,63** |
| 5 | Top Gun: Maverick | **7 160 804 869,01** |
| 6 | Barbie | **6 856 159 007,38** |
| 7 | The Super Mario Bros. Movie | **6 838 413 799,10** |
| 8 | The Lion King | **6 227 552

#### 1.2: Lucro médio por gênero (com receita informada)

In [50]:
pergunta = "Qual é o lucro médio por gênero, considerando apenas filmes com receita informada (receita_brl > 0)?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'list'}
[Passo 3] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_genres, bridge_movie_genre'}
[Passo 4] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT g.nome_genero,\n       AVG(f.lucro_brl) AS lucro_medio_brl\nFROM fact_movies_performance f\nJOIN bridge_movie_genre bg ON f.sk_movie_id = bg.sk_movie_id\nJOIN dim_genres g ON bg.sk_genre_id = g.sk_genre_id\nWHERE f.receita_brl > 0\nGROUP BY g.nome_genero\nORDER BY lucro_medio_brl DESC;'}



RESPOSTA DO AGENTE:
**Lucro médio por gênero (considerando apenas filmes com receita informada – receita_brl > 0)**  

| Gênero            | Lucro médio (BRL) |
|-------------------|-------------------|
| Science Fiction   | **520 783 718** |
| Adventure     

#### 1.3: Filmes com maior margem de lucro (receita e orçamento informados)

In [51]:
pergunta = "Quais são os filmes com maior margem de lucro (lucro dividido por receita ou orçamento), entre os que possuem receita e orçamento informados e maiores que zero?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT dm.titulo,\n       fmp.orcamento_usd,\n       fmp.receita_usd,\n       fmp.lucro_usd,\n       (fmp.lucro_usd / fmp.receita_usd) AS margem_lucro\nFROM dim_movies dm\nJOIN fact_movies_performance fmp ON dm.sk_movie_id = fmp.sk_movie_id\nWHERE fmp.orcamento_usd > 0\n  AND fmp.receita_usd > 0\nORDER BY margem_lucro DESC\nLIMIT 20;'}
[Passo 3] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT dm.titulo,\n       fmp.orcamento_usd,\n       fmp.receita_usd,\n       fmp.lucro_usd,\n       (fmp.lucro_usd * 1.0 / fmp.receita_usd) AS margem_receita,\n       (fmp.lucro_usd * 1.0 / fmp.orcamento_usd) AS margem_orcamento,\n       MAX(fmp.lucro_usd * 1.0 / fmp.receita_usd, fmp.lucro_usd * 1.0 / fmp.orcamento_usd) AS margem_maxima\nFROM dim_movies

### 2. Análise de Popularidade e Engajamento

#### 2.1: Os 5 filmes mais populares

In [52]:
pergunta = "Quais são os 5 filmes mais populares do catálogo segundo a métrica de popularidade?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT d.titulo, f.popularidade\nFROM fact_movies_performance f\nJOIN dim_movies d ON f.sk_movie_id = d.sk_movie_id\nORDER BY f.popularidade DESC\nLIMIT 5;'}



RESPOSTA DO AGENTE:
Os 5 filmes mais populares do catálogo, segundo a métrica **popularidade** (valor mais alto), são:

| Rank | Título | Popularidade |
|------|-------------------------------|--------------|
| 1 | **Blue Beetle** | 2 994,36 |
| 2 | **Gran Turismo** | 2 680,59 |
| 3 | **La Fellinette** | 2 020,00 |
| 4 | **The Fear Footage 2: Curse Of The Tape** | 2 019,00 |
| 5 | **Wwe Survivor Series 2018** | 2 018,00 |

Esses valores vêm da tabela de desempenho (`fact_movies_performance`) associada ao título na dimensão `dim_movies`.


#### 2.2: Filmes com maior divergência entre nota TMDB e nota IMDb

In [53]:
pergunta = "Quais são os filmes com maior divergência absoluta entre a nota TMDB e a nota IMDb?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT dm.titulo,\n       fmp.nota_tmdb,\n       fmp.nota_imdb,\n       ABS(fmp.nota_tmdb - fmp.nota_imdb) AS diff_abs\nFROM fact_movies_performance fmp\nJOIN dim_movies dm ON dm.sk_movie_id = fmp.sk_movie_id\nWHERE fmp.nota_tmdb IS NOT NULL AND fmp.nota_imdb IS NOT NULL\nORDER BY diff_abs DESC\nLIMIT 10;'}



RESPOSTA DO AGENTE:
**Filmes com maior divergência absoluta entre a nota TMDB e a nota IMDb**

| Rank | Título | Nota TMDB | Nota IMDb | Diferença absoluta |
|------|--------|-----------|-----------|--------------------|
| 1 | Soft Landing | 0,0 | 10,0 | **10,0** |
| 2 | Ayesha | 0,0 | 10,0 | **10,0** |
| 3 | The Process | 0,0 | 10,0 | **10,0** |
| 4 | Musica Pop | 0,0 | 10,0 | **10,0** |
| 5 | Makhlooq | 0,0 | 10,0 | **10,0** |
| 6 | Circular | 0,0 | 10,0 | **10,0*

#### 2.3: Nota média IMDb por ano de lançamento

In [54]:
pergunta = "Qual é a nota média IMDb dos filmes por ano de lançamento? Mostre os anos mais recentes."

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT dm.ano_lancamento AS ano, ROUND(AVG(fmp.nota_imdb), 2) AS nota_media_imdb\nFROM dim_movies dm\nJOIN fact_movies_performance fmp ON dm.sk_movie_id = fmp.sk_movie_id\nWHERE fmp.nota_imdb IS NOT NULL\nGROUP BY dm.ano_lancamento\nORDER BY dm.ano_lancamento DESC\nLIMIT 10;'}



RESPOSTA DO AGENTE:
**Nota média IMDb por ano de lançamento (anos mais recentes)**  

| Ano | Nota média IMDb |
|-----|-----------------|
| 2029 | 3,80 |
| 2027 | 6,40 |
| 2026 | 7,50 |
| 2025 | 6,58 |
| 2024 | 6,15 |
| 2023 | 6,23 |
| 2022 | 6,23 |
| 2021 | 6,23 |
| 2020 | 6,24 |
| 2019 | 6,26 |

*Os valores são a média das notas IMDb dos filmes lançados em cada ano, calculada a partir da camada Gold (tabelas `dim_movies` e `fact_movies_performance`).*


### 3. Análise de Elenco e Equipe

#### 3.1: Ator com mais participações nos últimos 5 anos

In [55]:
pergunta = "Qual é o ator (tipo_pessoa = 'Ator' em dim_people) com mais participações em filmes lançados nos últimos 5 anos?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance, dim_people, bridge_movie_person'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': "SELECT p.nome_pessoa AS ator, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes\nFROM dim_people p\nJOIN bridge_movie_person b ON p.sk_person_id = b.sk_person_id\nJOIN dim_movies m ON b.sk_movie_id = m.sk_movie_id\nWHERE p.tipo_pessoa = 'Ator'\n  AND m.data_lancamento >= DATE('now', '-5 years')\nGROUP BY p.sk_person_id, p.nome_pessoa\nORDER BY qtd_filmes DESC\nLIMIT 1;"}



RESPOSTA DO AGENTE:
O ator com o maior número de participações em filmes lançados nos últimos 5 anos é **Eric Roberts**, com **71 filmes** nesse período.


#### 3.2: Diretores com maior nota média (mínimo 5 filmes)

In [56]:
pergunta = "Quais são os diretores (tipo_pessoa = 'Diretor') com maior nota média IMDb, considerando apenas diretores com no mínimo 5 filmes dirigidos?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance, dim_people, bridge_movie_person'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': "SELECT p.nome_pessoa AS diretor,\n       COUNT(*) AS qtd_filmes,\n       AVG(f.nota_imdb) AS media_nota_imdb\nFROM bridge_movie_person b\nJOIN dim_people p ON b.sk_person_id = p.sk_person_id\nJOIN fact_movies_performance f ON b.sk_movie_id = f.sk_movie_id\nWHERE p.tipo_pessoa = 'Diretor'\nGROUP BY p.sk_person_id, p.nome_pessoa\nHAVING COUNT(*) >= 5\nORDER BY media_nota_imdb DESC;"}



RESPOSTA DO AGENTE:
**Diretores com maior nota média no IMDb (mínimo 5 filmes dirigidos)**  

| Rank | Diretor | Nº de filmes dirigidos | Nota média IMDb |
|------|------------------------|----------------------|-----------------|
| 1 | **Scott Wozniak** | 5 | 9,34 |
| 2 | **Yūichirō Hayashi** | 8 | 9,19 |
| 3 | **Jun Shishido** | 8 | 9,19 |
| 4 | **Trevor L.

#### 3.3: Dupla ator-diretor que mais trabalhou junta

In [57]:
pergunta = "Qual é a dupla formada por um ator e um diretor que mais trabalhou junta em filmes no catálogo?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_people, bridge_movie_person, dim_movies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': "WITH actor_director AS (\n    SELECT \n        mp_actor.sk_movie_id,\n        p_actor.nome_pessoa AS actor_name,\n        p_director.nome_pessoa AS director_name\n    FROM bridge_movie_person mp_actor\n    JOIN dim_people p_actor ON mp_actor.sk_person_id = p_actor.sk_person_id\n    JOIN bridge_movie_person mp_director ON mp_actor.sk_movie_id = mp_director.sk_movie_id\n    JOIN dim_people p_director ON mp_director.sk_person_id = p_director.sk_person_id\n    WHERE p_actor.tipo_pessoa = 'actor' AND p_director.tipo_pessoa = 'director'\n    AND p_actor.sk_person_id <> p_director.sk_person_id\n)\nSELECT actor_name, director_name, COUNT(DISTINCT sk_movie_id) AS movies_together\nFROM actor_director\nGROUP BY actor_name, director_name\nORDER BY mov

### 4. Análise de Gêneros e Produtoras

#### 4.1: Quantidade de filmes por gênero

In [58]:
pergunta = "Qual é a quantidade de filmes cadastrados para cada gênero no catálogo?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies'}
[Passo 2] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'list'}
[Passo 3] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_genres'}
[Passo 4] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT g.nome_genero AS genero, COUNT(DISTINCT b.sk_movie_id) AS quantidade_filmes\nFROM dim_genres g\nLEFT JOIN bridge_movie_genre b ON g.sk_genre_id = b.sk_genre_id\nGROUP BY g.nome_genero\nORDER BY quantidade_filmes DESC;'}



RESPOSTA DO AGENTE:
**Quantidade de filmes por gênero**

| Gênero            | Quantidade de filmes |
|-------------------|----------------------|
| Drama             | 28 086 |
| Documentary       | 18 082 |
| Comedy            | 16 048 |
| Horror            | 8 674 |
| Thriller          | 8 540 |
| Romance           | 6 209 |
| Action            | 5 028 |
| Animation  

#### 4.2: Produtora com maior lucro total

In [59]:
pergunta = "Qual é a produtora (dim_companies) com o maior lucro total acumulado em reais?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_companies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies'}
[Passo 3] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'bridge_movie_company'}
[Passo 4] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT dc.nome_produtora,\n       SUM(fmp.lucro_brl) AS lucro_total_brl\nFROM fact_movies_performance fmp\nJOIN bridge_movie_company bmc ON fmp.sk_movie_id = bmc.sk_movie_id\nJOIN dim_companies dc ON bmc.sk_company_id = dc.sk_company_id\nGROUP BY dc.nome_produtora\nORDER BY lucro_total_brl DESC\nLIMIT 1;'}



RESPOSTA DO AGENTE:
A produtora com o maior lucro total acumulado em reais é **Marvel Studios**, com um lucro acumulado de **R$ 63 936 626 417,88**.


#### 4.3: Gênero com maior margem de lucro média

In [60]:
pergunta = "Qual é o gênero cinematográfico com a maior margem de lucro média?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance'}
[Passo 2] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'list'}
[Passo 3] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_genres, bridge_movie_genre'}
[Passo 4] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'WITH movie_margin AS (\n    SELECT f.sk_movie_id,\n           CASE WHEN f.receita_usd > 0 THEN f.lucro_usd / f.receita_usd ELSE NULL END AS margem_lucro\n    FROM fact_movies_performance f\n    WHERE f.receita_usd > 0\n), genre_margin AS (\n    SELECT bg.sk_genre_id,\n           AVG(m.margem_lucro) AS avg_margem\n    FROM movie_margin m\n    JOIN bridge_movie_genre bg ON m.sk_movie_id = bg.sk_movie_id\n    GROUP BY bg.sk_genre_id\n)\nSELECT g.nome_genero,\n       ROUND(gm.avg_margem * 100, 2) AS margem_media_percentual\nFROM genre_margin gm\nJOIN dim_g

### 5. Análise de Avaliações dos Usuários

#### 5.1: Filmes mais avaliados pelos usuários

In [61]:
pergunta = "Quais são os filmes mais avaliados pelos usuários segundo a tabela de reviews (dim_reviews ou movie_reviews)?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_reviews, movie_reviews'}
[Passo 2] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies'}
[Passo 3] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT dm.titulo, dr.qtd_avaliacoes_usuarios, dr.nota_media_usuarios\nFROM dim_reviews dr\nJOIN dim_movies dm ON dr.sk_movie_id = dm.sk_movie_id\nORDER BY dr.qtd_avaliacoes_usuarios DESC\nLIMIT 10;'}
[Passo 4] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT dm.titulo,\n       SUM(dr.qtd_avaliacoes_usuarios) AS total_avaliacoes,\n       ROUND(AVG(dr.nota_media_usuarios),2) AS media_nota\nFROM dim_reviews dr\nJOIN dim_movies dm ON dr.sk_movie_id = dm.sk_movie_id\nGROUP BY dm.titulo\nORDER BY total_avaliacoes DESC\nLIMIT 10;'}



RESPOSTA DO AGENTE:
**Top 10 filmes mais avaliados pelos usuários (tabela `dim_reviews`)**

| Rank | Filme | Total d

#### 5.2: Filmes em que a nota dos usuários mais diverge da nota IMDb

In [62]:
pergunta = "Quais são os filmes em que a nota média dos usuários mais diverge da nota IMDb?"

resposta = run_cinedata_agente(pergunta)
print("\n\n")
print("RESPOSTA DO AGENTE:")
print(resposta)


[Passo 1] O agente chamou essa tool: describe_tables com esses argumentos: {'table_names': 'dim_movies, fact_movies_performance, dim_people, bridge_movie_person'}
[Passo 2] O agente chamou essa tool: execute_sql_query com esses argumentos: {'query': 'SELECT dm.titulo, f.nota_tmdb, f.nota_imdb, ABS(f.nota_tmdb - f.nota_imdb) AS diff\nFROM dim_movies dm\nJOIN fact_movies_performance f USING (sk_movie_id)\nORDER BY diff DESC\nLIMIT 10;'}



RESPOSTA DO AGENTE:
**Filmes com maior divergência entre a nota média dos usuários (TMDB) e a nota IMDb**

| Rank | Título | Nota TMDB (média usuários) | Nota IMDb | Diferença absoluta |
|------|---------------------------|---------------------------|-----------|-------------------|
| 1 | Soft Landing | 0,0 | 10,0 | **10,0** |
| 2 | Ayesha | 0,0 | 10,0 | **10,0** |
| 3 | The Process | 0,0 | 10,0 | **10,0** |
| 4 | Musica Pop | 0,0 | 10,0 | **10,0** |
| 5 | Makhlooq | 0,0 | 10,0 | **10,0** |
| 6 | Circular | 0,0 | 10,0 | **10,0** |
| 7 | Twin Cities Pri